# Problem: Write a Custom Activation Function

### Problem Statement
You are tasked with implementing a **custom activation function** in PyTorch that computes the following operation: 
 
$ \text{activation}(x) = \tanh(x) + x $ 

Once implemented, this custom activation function will be used in a simple linear regression model.

### Requirements
1. **Custom Activation Function**:
   - In the class `CustomActivationModel` (an `nn.Module`), implement the method `custom_activation(x)` that returns $\tanh(x) + x$ elementwise.

2. **Integration with Linear Regression**:
   - The model has a single linear layer, `self.linear = nn.Linear(1, 1)` (already provided).
   - Its `forward(x)` applies the custom activation to the output of that layer: $\hat{y} = \tanh(wx + b) + (wx + b)$.

### Constraints
- The custom activation function should not have any learnable parameters.
- Build it from differentiable torch ops so autograd can backpropagate through it (no Python `if` on tensor values).

<details>
  <summary>💡 Hint</summary>
  Some details: https://stackoverflow.com/questions/55765234/pytorch-custom-activation-functions
</details>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib import pyplot as plt

In [2]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

In [ ]:
# Define the Linear Regression Model within a CustomActivationModel class
class CustomActivationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # Single input and single output

    def custom_activation(self, x):
        # No learnable parameters; built from torch ops, so autograd handles the backward pass
        return torch.tanh(x) + x

    def forward(self, x):
        return self.custom_activation(self.linear(x))

# Initialize the model, loss function, and optimizer
model = CustomActivationModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}")

# Plot the model fit to the train data. The fitted curve is the model's output,
# tanh(wx + b) + (wx + b) — not the bare line wx + b, which sits about 1 lower
# wherever tanh has saturated.
order = X[:, 0].argsort()
with torch.no_grad():
    fit = model(X[order])
plt.figure(figsize=(4, 4))
plt.scatter(X, y, label='Training Data')
plt.plot(X[order], fit, 'r', label='Model Fit')
plt.legend()
plt.show()

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")